# 11 -- Quantum STEM: Scanning Transmission Electron Microscopy

STEM scans a focused electron probe across the sample and, at each position,
integrates the scattered intensity over annular detectors:

- **HAADF** (high-angle annular dark field, Z-contrast)
- **ADF** / **ABF** (annular/annular-bright field)
- **BF** (bright field)

Unlike CTEM (one circuit, one image), STEM runs **one quantum circuit per probe
position** -- that per-position cost is the central practical constraint of
everything in this notebook.

## Worth understanding before you tune parameters:

Detector angles are specified in **mrad**, and convert to spatial frequency via
$k\,[\text{\AA}^{-1}] = \text{mrad}\times 10^{-3}/\lambda$. At 200 kV,
$\lambda\approx 0.023\,\text{\AA}$ -- tiny -- so HAADF's 60-200 mrad band alone
needs $k$ up to ~$8.6 \text{\AA}^{-1}$. Your **Nyquist limit** is
$k_{max} = 1/(2\,\Delta x)$, set by pixel size. If your grid is too coarse for
the field of view you're using, the HAADF/ADF masks can end up **entirely
outside** the reachable $k$-space -- giving a flat, contrast-free image, not a
subtle numerical artifact. The fix is either a smaller field of view (fewer
unit cells, sampled finely) or more pixels -- both cost qubits
($n_q = 2\log_2 N$), so there's a real spatial-extent-vs-angular-reach
tradeoff at any fixed qubit budget.

This notebook uses a **small field of view** (2x1 MoS2 unit cells) so the demo
runs in seconds; the markdown cells along the way show the math for scaling up
to match CTEM's full field of view if you want a real comparison (that costs
qubits and runtime -- see the note near the end).

In [ ]:
%matplotlib inline
import time
import numpy as np
import matplotlib.pyplot as plt

from quscope.quantum_ctem.quantum_ctem_circuit import relativistic_wavelength
from quscope.quantum_ctem.quantum_stem import STEMDetectors
from quscope.quantum_ctem.quantum_stem_multislice import run_stem_multislice

VOLTAGE = 200e3
CS_MM = 1.3


## Building a small, finely-sampled STEM patch

Same atom basis/generator as notebook 01, just a much smaller
`n_cells_x`/`n_cells_y` so the pixel size shrinks and $k$-space reaches
realistic detector angles.

In [ ]:
def build_mos2_supercell_potential(grid_size, n_cells_x=5, n_cells_y=3):
    a = 3.18
    b_lat = a * np.sqrt(3.0)
    L = max(n_cells_x * a, n_cells_y * b_lat)
    px = L / grid_size
    coords = np.linspace(0.0, L, grid_size, endpoint=False)
    X, Y = np.meshgrid(coords, coords, indexing="ij")
    V = np.zeros((grid_size, grid_size))
    atom_basis = {
        "Mo": {"frac": [(0.0, 0.0), (1/3, 1/3)], "amp": 600.0, "width": 0.35},
        "S":  {"frac": [(0.0, 1/6), (0.0, -1/6), (1/3, 1/2), (1/3, 1/6)],
               "amp": 300.0, "width": 0.25},
    }
    for element, info in atom_basis.items():
        amp, width = info["amp"], info["width"]
        for fx, fy in info["frac"]:
            for icx in range(-1, n_cells_x + 1):
                for icy in range(-1, n_cells_y + 1):
                    xc, yc = (icx + fx) * a, (icy + fy) * b_lat
                    if -1.0 <= xc <= L + 1.0 and -1.0 <= yc <= L + 1.0:
                        V += amp * np.exp(-((X-xc)**2 + (Y-yc)**2) / (2*width**2))
    return V, px


STEM_GRID = 64                 # n_qubits = 12 -- fast for this demo
STEM_N_CELLS_X, STEM_N_CELLS_Y = 2, 1
STEM_CONVERGENCE_MRAD = 15.0
STEM_SCAN_STEP_PX = 8          # coarse scan for a quick demo

V_stem, px_stem = build_mos2_supercell_potential(
    STEM_GRID, n_cells_x=STEM_N_CELLS_X, n_cells_y=STEM_N_CELLS_Y)

lam = relativistic_wavelength(VOLTAGE)
k_max = 1 / (2 * px_stem)
print(f"STEM patch: {STEM_N_CELLS_X}x{STEM_N_CELLS_Y} unit cells, "
      f"pixel_size={px_stem:.4f} \u00c5")
print(f"Nyquist reaches {k_max*lam*1000:.0f} mrad "
      f"(probe convergence = {STEM_CONVERGENCE_MRAD} mrad, well below that)")


## Section 1: Single-slice STEM (WPOA)

`run_stem_multislice(..., n_slices=1)` reproduces single-slice WPOA STEM. It
does **not** use `QuantumCircuit.initialize()` to inject the probe state
(that call scales as $O(4^n)$ and becomes impractical above ~12-13 qubits);
instead the probe array is handed straight to a `Statevector` and only the
gate portion of the circuit is applied via `.evolve()` -- mathematically
identical, dramatically faster.

In [ ]:
def plot_stem_panel(result, title_prefix=""):
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    channels = ["HAADF", "ADF", "ABF", "BF"]
    cmaps = ["inferno", "inferno", "gray", "gray"]
    for ax, ch, cm in zip(axes, channels, cmaps):
        im = ax.imshow(result[ch], cmap=cm, origin="lower")
        ax.set_title(f"{title_prefix}{ch}", fontsize=11)
        ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(result["metrics"]["approach"])
    plt.tight_layout()
    plt.show()


t0 = time.time()
result_single = run_stem_multislice(
    V_stem, pixel_size=px_stem, voltage=VOLTAGE, n_slices=1,
    convergence_mrad=STEM_CONVERGENCE_MRAD, cs_mm=CS_MM,
    detectors=STEMDetectors(), scan_step_px=STEM_SCAN_STEP_PX,
)
print(f"took {time.time()-t0:.2f}s")
plot_stem_panel(result_single)


## Section 3: Quantum multislice STEM

The probe is propagated through several slices, alternating quantum phase
gratings with QFT-based Fresnel propagation between them -- the STEM analogue
of the multislice CTEM in notebook 01.

Performance note: diagonal gates (phase gratings, Fresnel propagators) are
applied as exact elementwise array multiplication rather than routed through
Qiskit's `DiagonalGate` synthesis (mathematically identical, ~1000x faster at
these qubit counts); the `QFTGate` steps -- the genuinely quantum part of the
pipeline -- still run as real Qiskit circuits via `Statevector.evolve()`.

In [ ]:
t0 = time.time()
result_ms = run_stem_multislice(
    V_stem, pixel_size=px_stem, voltage=VOLTAGE, n_slices=4,
    slice_thickness=6.5, convergence_mrad=STEM_CONVERGENCE_MRAD, cs_mm=CS_MM,
    detectors=STEMDetectors(), scan_step_px=STEM_SCAN_STEP_PX,
)
print(f"took {time.time()-t0:.2f}s, {result_ms['metrics']['approach']}")
plot_stem_panel(result_ms, title_prefix="4-slice ")
